# 06 Segmentation

Flood-It! retention and churn project. Run cells top to bottom.

### Imports and settings

In [ ]:
import pandas as pd                                                    # tables
import numpy as np                                                     # maths
import matplotlib.pyplot as plt                                        # charts
import seaborn as sns                                                  # nicer charts
from scipy import stats                                                # chi-square test
from sklearn.preprocessing import StandardScaler                       # put features on the same scale
from sklearn.cluster import KMeans                                     # clustering algorithm
from sklearn.metrics import silhouette_score, adjusted_rand_score      # cluster quality and stability
from sklearn.decomposition import PCA                                  # squeeze features into 2D for plotting
from pathlib import Path                                               # file paths

sns.set_theme(style="whitegrid")                                       # chart style
pd.set_option("display.max_columns", None)                             # never hide columns
pd.set_option("display.width", 200)                                    # wide printing

### Paths, data and segmentation features

In [ ]:
ROOT = Path.cwd()                                                      # folder the notebook runs in
if ROOT.name == "notebooks":                                           # if inside notebooks/...
    ROOT = ROOT.parent                                                 # ...go up to the project root
DATA = ROOT / "data" / "processed"                                     # data folder
FIG = ROOT / "reports" / "figures"                                     # charts folder
players = pd.read_csv(DATA / "player_features.csv")                    # one row per new player
players["d0_completion_rate"] = np.where(players["d0_levels_started"] > 0, players["d0_levels_completed"] / players["d0_levels_started"], 0.0)   # share of levels completed
SEG_FEATURES = ["d0_engaged_minutes", "d0_sessions", "d0_levels_started", "d0_completion_rate", "d0_ad_rewards", "d0_currency_spends"]   # behaviour only, no outcome columns
X = players[SEG_FEATURES].copy()                                       # feature table
for col in ["d0_engaged_minutes", "d0_sessions", "d0_levels_started", "d0_ad_rewards", "d0_currency_spends"]:   # one loop over count-like columns
    X[col] = np.log1p(X[col])                                          # log scale so heavy players do not dominate
X_scaled = StandardScaler().fit_transform(X)                           # mean 0, standard deviation 1 for every feature
print(X_scaled.shape)                                                  # rows and columns

### Choose the number of clusters: elbow and silhouette

In [ ]:
rng = np.random.default_rng(7)                                         # seed for the sample below
sample_idx = rng.choice(len(X_scaled), size=min(3000, len(X_scaled)), replace=False)   # silhouette is slow, so score a sample
rows = []                                                              # results collector
for k in range(2, 9):                                                  # one loop, try k = 2 ... 8
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)   # fit k-means
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(X_scaled[sample_idx], km.labels_[sample_idx])})   # tightness and separation
choice = pd.DataFrame(rows)                                            # to table
print(choice.round(4).to_string(index=False))                          # show
fig, axes = plt.subplots(1, 2, figsize=(12, 4))                        # two charts
axes[0].plot(choice["k"], choice["inertia"], marker="o")               # elbow chart
axes[0].set_title("Elbow: inertia by k")                               # title
axes[1].plot(choice["k"], choice["silhouette"], marker="o", color="tab:orange")   # silhouette chart
axes[1].set_title("Silhouette score by k (higher = better separated)")   # title
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "16_choose_k.png", dpi=150)                          # save
plt.show()                                                             # display

### Fit the final model

In [ ]:
candidates = choice[(choice["k"] >= 3) & (choice["k"] <= 6)]           # 3-6 segments are practical for a game team
K = int(candidates.loc[candidates["silhouette"].idxmax(), "k"])        # best-separated option in that range
print("Chosen K:", K, "(confirm with the elbow chart before accepting)")   # explain the choice
final = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_scaled)   # final model with more restarts
players["segment"] = final.labels_                                     # attach segment numbers

### Profile each segment and give it a readable name

In [ ]:
profile = players.groupby("segment").agg(
    players=("player_id", "count"),                                    # size
    median_minutes=("d0_engaged_minutes", "median"),                   # typical engagement
    median_sessions=("d0_sessions", "median"),                         # typical sessions
    median_levels_started=("d0_levels_started", "median"),             # typical progress
    avg_completion_rate=("d0_completion_rate", "mean"),                # skill / difficulty signal
    avg_ad_rewards=("d0_ad_rewards", "mean"),                          # rewarded-ad usage
    d1_retention=("retained_d1", "mean"),                              # outcome (NOT used for clustering)
    returned_within_7d=("returned_within_7d", "mean"),                 # outcome (NOT used for clustering)
)
profile["share_of_players"] = profile["players"] / profile["players"].sum()   # percentage of all players
order = profile["median_levels_started"].rank(method="first").astype(int)   # 1 = least active segment
base_names = ["Quick bouncers", "Light samplers", "Steady players", "Keen progressors", "Power players", "Superfans"]   # starting names, least to most active
profile["suggested_name"] = [base_names[r - 1] for r in order]         # rename after reading the profile
players["segment_name"] = players["segment"].map(profile["suggested_name"])   # attach names to players
print(profile.round(3).sort_values("median_levels_started").to_string())   # least to most active

### Stability check: do different random starts give the same segments?

In [ ]:
aris = []                                                              # agreement scores
for seed in [1, 2, 3, 4, 5]:                                           # one loop, one refit per seed
    labels = KMeans(n_clusters=K, n_init=10, random_state=seed).fit_predict(X_scaled)   # refit with another seed
    aris.append(adjusted_rand_score(final.labels_, labels))            # 1.0 = identical grouping
print("Adjusted Rand Index vs final model:", np.round(aris, 3), "| mean:", round(np.mean(aris), 3))

### Is return rate different across segments?

In [ ]:
table = pd.crosstab(players["segment_name"], players["returned_within_7d"])   # counts
chi2, p, dof, _ = stats.chi2_contingency(table)                        # independence test
print(f"Chi-square = {chi2:.1f}, dof = {dof}, p = {p:.3g}")            # result
fig, ax = plt.subplots(figsize=(9, 4))                                 # empty chart
rates = players.groupby("segment_name")["returned_within_7d"].mean().sort_values()   # rate per segment
ax.barh(rates.index, 100 * rates.values, color="tab:green")            # bars
ax.set_title("Returned within 7 days by player segment")               # title
ax.set_xlabel("%")                                                     # x label
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "17_segment_retention.png", dpi=150)                 # save
plt.show()                                                             # display

### Chart: segments in two dimensions (PCA)

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(X_scaled[sample_idx])   # 2D projection of the sample
fig, ax = plt.subplots(figsize=(8, 6))                                 # empty chart
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=players["segment_name"].to_numpy()[sample_idx], s=12, alpha=0.6, ax=ax)   # points coloured by segment
ax.set_title("Player segments projected onto two principal components")   # title
ax.set_xlabel("PC1")                                                   # x label
ax.set_ylabel("PC2")                                                   # y label
fig.tight_layout()                                                     # tidy
fig.savefig(FIG / "18_segments_pca.png", dpi=150)                      # save
plt.show()                                                             # display

### Save segments

In [ ]:
players[["player_id", "segment", "segment_name"]].to_csv(DATA / "player_segments.csv", index=False)   # for the dashboard
profile.to_csv(DATA / "segment_profiles.csv")                          # for the memo
print("Saved segments")                                                # confirm